# NOTEBOOK 04: FEATURE ENGINEERING, CONSTRUCTION & ANTI-LEAKAGE
## Computational Society Intelligence Platform (CSIP) — Modul MVP: BDI
**Kepatuhan SKKNI**:
- `[UK-05 Menentukan Objek Data]` (4 Objek Data Analitik)
- `[UK-07 Mengkonstruksi Data]` (Rekayasa Fitur & Anti-Leakage Protocol)

---

### 1. Tujuan Rekayasa Fitur (UK-07)
1. **Konstruksi Fitur Ekonomi**: `total_item_price`, `total_freight`, `freight_ratio`, `item_count`.
2. **Konstruksi Fitur Pengalaman Logistik**: `delivery_lead_time_days`, `delivery_delay_days`, `is_late`, `approval_lag_hours`.
3. **Konstruksi Jarak Geospasial Haversine**: `seller_buyer_distance_km` antara kode pos pembeli dan penjual.
4. **Konstruksi Fitur Interaksi Non-Linear**: `delay_x_freight_ratio`, `distance_x_weight`, `high_value_late_flag`.
5. **Penegakan Protokol Anti-Leakage Mutlak (Zero Data Leakage Guarantee)**:
   - *Point-in-Time Correctness*: Fitur checkout tidak mengandung informasi masa depan.
   - *First-Order Isolation*: Fitur prediksi retensi hanya bersumber dari Transaksi Pertama pelanggan.
   - *Split-Before-Transform*: Skala/imputasi di-fit HANYA pada training split.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Load 4 constructed processed datasets
order_level = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "order_level.parquet")
customer_cohort = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "customer_cohort.parquet")
review_corpus = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "review_corpus.parquet")
weekly_state = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "weekly_system_state.parquet")

print("4 Processed Datasets Loaded Successfully:")
print(f"  1. Order-Level       : {len(order_level):,} rows x {len(order_level.columns)} cols")
print(f"  2. Customer-Cohort   : {len(customer_cohort):,} rows x {len(customer_cohort.columns)} cols")
print(f"  3. Review-Corpus     : {len(review_corpus):,} rows x {len(review_corpus.columns)} cols")
print(f"  4. Weekly System State: {len(weekly_state):,} rows x {len(weekly_state.columns)} cols")


4 Processed Datasets Loaded Successfully:
  1. Order-Level       : 96,470 rows x 41 cols
  2. Customer-Cohort   : 75,318 rows x 23 cols
  3. Review-Corpus     : 95,824 rows x 9 cols
  4. Weekly System State: 87 rows x 10 cols


### 2. Telaah Fitur Order-Level & Interaksi Non-Linear
Menampilkan statistik deskriptif dari fitur ekonomi, pengalaman, dan interaksi yang telah direkayasa.


In [2]:
feature_cols = [
    "total_order_value", "freight_ratio", "delivery_lead_time_days",
    "delivery_delay_days", "seller_buyer_distance_km",
    "delay_x_freight_ratio", "distance_x_weight", "is_late", "negative_review_flag"
]
display(order_level[feature_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(2))


,total_order_value,freight_ratio,delivery_lead_time_days,delivery_delay_days,seller_buyer_distance_km,delay_x_freight_ratio,distance_x_weight,is_late,negative_review_flag
count,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00
mean,159.83,0.31,12.56,-11.18,599.36,0.24,4018.85,0.08,0.13
std,218.80,0.31,9.55,10.18,592.20,1.96,4096.24,0.27,0.33
min,9.59,0.00,0.53,-146.02,0.00,0.00,0.00,0.00,0.00
25%,61.85,0.13,6.77,-16.24,182.81,0.00,1233.23,0.00,0.00
50%,105.28,0.22,10.22,-11.95,432.57,0.00,2841.58,0.00,0.00
75%,176.26,0.38,15.72,-6.39,796.88,0.00,5153.91,0.00,0.00
95%,446.26,0.84,29.27,3.82,2095.11,0.72,13306.83,1.00,1.00
max,13664.08,21.45,209.63,188.98,3398.49,150.55,30848.01,1.00,1.00


### 3. Verifikasi Zero Data Leakage pada Customer Cohort
Memverifikasi bahwa prediktor retensi (`repeat_purchase_flag`) **hanya** menggunakan fitur Transaksi Pertama (*Order 1 Isolation*), bebas dari agregat seumur hidup (*lifetime spend / count*) yang membocorkan label masa depan.


In [3]:
# Cek ketiadaan fitur terlarang
prohibited_features = ["lifetime_order_count", "lifetime_total_spend", "lifetime_avg_review"]
for col in prohibited_features:
    assert col not in customer_cohort.columns, f"LEAKAGE DETECTED: {col} ada di customer_cohort!"

print("Status Anti-Leakage: VERIFIED (Zero Prohibited Lifetime Features)")
print(f"Tingkat Repeat Purchase (Jendela Observasi Delta_W = 90 hari): {customer_cohort['repeat_purchase_flag'].mean()*100:.2f}%")


Status Anti-Leakage: VERIFIED (Zero Prohibited Lifetime Features)
Tingkat Repeat Purchase (Jendela Observasi Delta_W = 90 hari): 2.28%


### 4. Skema Out-of-Time Temporal Split
Membagi data berdasarkan urutan waktu nyata transaksi:
- **Train Split**: 2017-01-01 s/d 2018-03-31
- **Validation Split**: 2018-04-01 s/d 2018-06-30
- **Locked Test Split**: 2018-07-01 s/d 2018-08-31


In [4]:
train_mask = (order_level["order_purchase_timestamp"] >= "2017-01-01") & (order_level["order_purchase_timestamp"] <= "2018-03-31")
val_mask = (order_level["order_purchase_timestamp"] >= "2018-04-01") & (order_level["order_purchase_timestamp"] <= "2018-06-30")
test_mask = (order_level["order_purchase_timestamp"] >= "2018-07-01") & (order_level["order_purchase_timestamp"] <= "2018-08-31")

print(f"Train Split Rows     : {train_mask.sum():,} ({train_mask.sum()/len(order_level)*100:.1f}%)")
print(f"Validation Split Rows: {val_mask.sum():,} ({val_mask.sum()/len(order_level)*100:.1f}%)")
print(f"Locked Test Rows     : {test_mask.sum():,} ({test_mask.sum()/len(order_level)*100:.1f}%)")


Train Split Rows     : 63,889 (66.2%)
Validation Split Rows: 19,520 (20.2%)
Locked Test Rows     : 12,507 (13.0%)


### 5. Kesimpulan Konstruksi Data
Konstruksi fitur telah selesai dan memenuhi **Validation Gate G3**:
1. 4 Objek Data Analitik telah tersimpan dalam format Parquet terkompresi di `data/processed/`.
2. Seluruh fitur telah bebas dari nilai null yang tak tertangani.
3. Protokol anti-leakage terverifikasi 100% lulus.
